# 01 — Public wedge: lineage on jaffle

This notebook uses the committed [`examples/lineage-demo`](../lineage-demo) project.
It shows the **public product promise**: merge warehouse metadata + dbt into one graph,
then compile, explore nodes, and trace column impact.

No live warehouse connection — only local fixture files referenced in `clearmetric.yaml`.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import wedge_project

PROJECT_DIR = wedge_project()
assert (PROJECT_DIR / "clearmetric.yaml").is_file(), PROJECT_DIR
print(f"project: {PROJECT_DIR}")

## Compile the enforced graph

`compile()` runs ingest → merge → bind → enforce (cleaner + security floor).
`build_graph()` stops before enforce — useful when you want a report first.

In [ ]:
from clearmetric.compiler import build_graph, check_graph, compile

built = build_graph(PROJECT_DIR)
report = check_graph(built.artifact, posture=built.project.posture)
errors = sum(1 for f in report.findings if f.severity == "error")
warnings = sum(1 for f in report.findings if f.severity == "warn")
print(f"nodes: {len(built.artifact.nodes)}  edges: {len(built.artifact.edges)}")
print(f"clean findings — errors: {errors}  warnings: {warnings}")

compiled = compile(PROJECT_DIR)
artifact = compiled.artifact

## Explore node kinds

The full graph includes models, columns, tables, and lineage edges. Catalog output
(notebook 02) slices to table/column/model only.

In [ ]:
kind_counts = Counter(node.kind for node in artifact.nodes)
for kind, count in sorted(kind_counts.items()):
    print(f"{kind:10} {count}")

orders_amount = next(n for n in artifact.nodes if n.id == "column:orders.amount")
print(
    "\ncolumn:orders.amount bindings:",
    json.dumps(orders_amount.bindings, indent=2, default=str),
)

## Impact: what is upstream of `orders.amount`?

Same selection the CLI accepts: `orders.amount`, `column:orders.amount`, etc.

In [ ]:
from clearmetric.compiler.impact import impact

compiled, result = impact(
    PROJECT_DIR,
    selection="orders.amount",
    direction="upstream",
)
print(f"selection_id: {result.selection_id}")
print(f"related_ids ({len(result.related_ids)}):")
for node_id in result.related_ids:
    print(f"  {node_id}")
if result.warnings:
    print("\nwarnings:")
    for warning in result.warnings:
        print(f"  [{warning.code}] {warning.message}")

## CLI equivalent

The CLI wraps the same compiler pipeline. Useful when scripting from shell or CI.

In [ ]:
from clearmetric.cli.runner import run_cm

scan = run_cm(PROJECT_DIR, "scan")
print(scan.stdout)

impact_cli = run_cm(
    PROJECT_DIR,
    "impact",
    "orders.amount",
    "--upstream",
    "--format",
    "json",
)
assert impact_cli.returncode == 0, impact_cli.stderr
payload = json.loads(impact_cli.stdout)
print(f"CLI selection_id: {payload['selection_id']}")